In [0]:
%sql
DESCRIBE jarvis_etl.bronze.transactions;


col_name,data_type,comment
id,int,null
date,timestamp,null
client_id,int,null
card_id,int,null
amount,string,null
use_chip,string,null
merchant_id,int,null
merchant_city,string,null
merchant_state,string,null
zip,double,null


In [0]:
%sql
select * from jarvis_etl.bronze.transactions limit 5;


id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors
7475327,2010-01-01T00:01:00.000Z,1556,2972,null,Swipe Transaction,59935,Beulah,ND,58523.0,5499,
7475328,2010-01-01T00:02:00.000Z,561,4575,null,Swipe Transaction,67570,Bettendorf,IA,52722.0,5311,
7475329,2010-01-01T00:02:00.000Z,1129,102,null,Swipe Transaction,27092,Vista,CA,92084.0,4829,
7475331,2010-01-01T00:05:00.000Z,430,2860,null,Swipe Transaction,27092,Crown Point,IN,46307.0,4829,
7475332,2010-01-01T00:06:00.000Z,848,3915,null,Swipe Transaction,13051,Harwood,MD,20776.0,5813,


In [0]:
%sql
DESCRIBE jarvis_etl.bronze.cards;

col_name,data_type,comment
id,smallint,null
client_id,smallint,null
card_brand,string,null
card_type,string,null
card_number,bigint,null
expires,string,null
cvv,smallint,null
has_chip,boolean,null
num_cards_issued,smallint,null
credit_limit,"decimal(19,4)",null


In [0]:
%sql
DESCRIBE jarvis_etl.bronze.users;

col_name,data_type,comment
id,int,null
current_age,int,null
retirement_age,int,null
birth_year,int,null
birth_month,int,null
gender,string,null
address,string,null
latitude,double,null
longitude,double,null
per_capita_income,string,null


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS jarvis_etl.silver;

In [0]:
from pyspark.sql.functions import col, regexp_replace, trim, to_timestamp
from pyspark.sql.types import DecimalType, IntegerType

transactions_bronze = spark.table("jarvis_etl.bronze.transactions")
mcc_bronze = spark.table("jarvis_etl.bronze.mcc_codes")
fraud_bronze = spark.table("jarvis_etl.bronze.fraud_labels")

transactions_clean = (
    transactions_bronze
    .withColumn("amount", regexp_replace(col("amount"), "\\$", "").cast(DecimalType(10, 2)))
    .withColumn("transaction_date", to_timestamp(col("date")))
    .drop("date")
    .filter(col("amount").isNotNull())
)

transactions_enriched = (
    transactions_clean
    .join(fraud_bronze, transactions_clean.id == fraud_bronze.transaction_id, "left")
    .drop("transaction_id")
    .join(mcc_bronze, transactions_clean.mcc == mcc_bronze.mcc_code, "left")
    .drop("mcc_code")
)

transactions_enriched.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("jarvis_etl.silver.transactions")
print(f"silver transactions row count: {transactions_enriched.count()}")

silver transactions row count: 13305915


In [0]:
spark.sql("Select * from jarvis_etl.silver.transactions limit 10")

DataFrame[id: int, client_id: int, card_id: int, amount: decimal(10,2), use_chip: string, merchant_id: int, merchant_city: string, merchant_state: string, zip: double, mcc: int, errors: string, transaction_date: timestamp, is_fraud: string, description: string]

In [0]:
%sql
select * from jarvis_etl.silver.transactions limit 10

id,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors,transaction_date,is_fraud,description
7475501,526,5917,22.23,Online Transaction,16798,ONLINE,null,null,4121,null,2010-01-01T04:31:00.000Z,No,Taxicabs and Limousines
7475525,1964,2508,18.24,Swipe Transaction,715,Clayton,NC,27520.0,7230,null,2010-01-01T05:12:00.000Z,null,Beauty and Barber Shops
7476101,1591,2043,80.00,Swipe Transaction,59935,Omaha,NE,68106.0,5499,null,2010-01-01T08:17:00.000Z,No,Miscellaneous Food Stores
7476246,416,2399,39.08,Swipe Transaction,2414,Nashville,TN,37211.0,7538,null,2010-01-01T08:57:00.000Z,null,Automotive Service Shops
7476923,291,169,10.23,Swipe Transaction,47090,Dalton,GA,30720.0,5812,null,2010-01-01T11:22:00.000Z,No,Eating Places and Restaurants
7476958,368,5566,31.81,Online Transaction,50404,ONLINE,null,null,4784,null,2010-01-01T11:30:00.000Z,No,Tolls and Bridge Fees
7477221,1962,2126,18.21,Swipe Transaction,50783,Lyons,IL,60534.0,5411,null,2010-01-01T12:18:00.000Z,No,"Grocery Stores, Supermarkets"
7477483,0,4639,33.96,Swipe Transaction,20519,Portland,ME,4101.0,5942,null,2010-01-01T13:10:00.000Z,No,Book Stores
7477790,1797,1127,32.69,Swipe Transaction,33326,Kahului,HI,96732.0,4121,null,2010-01-01T14:19:00.000Z,No,Taxicabs and Limousines
7477902,450,5176,73.00,Swipe Transaction,50783,Newport News,VA,23606.0,5411,null,2010-01-01T14:49:00.000Z,null,"Grocery Stores, Supermarkets"


In [0]:
from pyspark.sql.functions import to_date

cards_bronze = spark.table("jarvis_etl.bronze.cards")

cards_clean = (
    cards_bronze
    .withColumn("credit_limit", col("credit_limit").cast(DecimalType(10, 2)))
    .withColumn("acct_open_date", to_date(col("acct_open_date"), "MM/yyyy"))
)

cards_clean.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("jarvis_etl.silver.cards")
print(f"silver cards row count: {cards_clean.count()}")

silver cards row count: 6146


In [0]:
users_bronze = spark.table("jarvis_etl.bronze.users")

users_clean = (
    users_bronze
    .withColumn("per_capita_income", regexp_replace(col("per_capita_income"), "\\$|,", "").cast(DecimalType(12, 2)))
    .withColumn("yearly_income", regexp_replace(col("yearly_income"), "\\$|,", "").cast(DecimalType(12, 2)))
    .withColumn("total_debt", regexp_replace(col("total_debt"), "\\$|,", "").cast(DecimalType(12, 2)))
)

users_clean.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("jarvis_etl.silver.users")
print(f"silver users row count: {users_clean.count()}")

silver users row count: 2000


In [0]:
%sql
SELECT 'transactions' as table_name, COUNT(*) as row_count FROM jarvis_etl.silver.transactions
UNION ALL
SELECT 'cards', COUNT(*) FROM jarvis_etl.silver.cards
UNION ALL
SELECT 'users', COUNT(*) FROM jarvis_etl.silver.users;

table_name,row_count
transactions,13305915
cards,6146
users,2000
